# 3.7 File Handling

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 3: Python for Machine Learning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-03-python-for-machine-learning/03.07_file-handling.ipynb)

Companion code for Section 3.7 of the book. Run the cells in order from top to bottom.

**Example 3.7.1**

In [ ]:
# Writing a text file - context manager ensures file is closed even if error occurs
training_log = [
    'epoch,loss,val_loss,accuracy,val_accuracy',
    '1,0.9123,0.9456,0.6234,0.6012',
    '2,0.7891,0.8234,0.7123,0.6934',
    '3,0.6532,0.7012,0.7891,0.7534',
]

with open('training_log.txt', 'w') as f:
    for line in training_log:
        f.write(line + '\n')
print('Training log written.')

# Reading entire file
with open('training_log.txt', 'r') as f:
    content = f.read()
print('Full content:')
print(content)

# Reading line by line (memory-efficient for large files)
with open('training_log.txt', 'r') as f:
    header = f.readline().strip()           # First line = header
    columns = header.split(',')             # Parse column names
    print(f'Columns: {columns}')

# Using pathlib - modern file path handling
from pathlib import Path

log_path = Path('training_log.txt')
print(f'Exists: {log_path.exists()}')
print(f'Size:   {log_path.stat().st_size} bytes')
print(f'Stem:   {log_path.stem}')          # 'training_log' (no extension)
print(f'Suffix: {log_path.suffix}')        # '.txt'

# Read all text at once (pathlib)
text = log_path.read_text()
lines = text.strip().split('\n')
print(f'Lines: {len(lines)}')

**Example 3.7.2**

In [ ]:
import csv
# Writing CSV
results = [
    {'model': 'LogReg',   'accuracy': 0.87, 'f1': 0.865, 'time_s': 0.5},
    {'model': 'XGBoost',  'accuracy': 0.94, 'f1': 0.938, 'time_s': 12.3},
    {'model': 'ResNet50', 'accuracy': 0.97, 'f1': 0.971, 'time_s': 450.0},
]

with open('model_results.csv', 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['model','accuracy','f1','time_s'])
    writer.writeheader()
    writer.writerows(results)

# Reading CSV
with open('model_results.csv', 'r') as f:
    reader = csv.DictReader(f)
    for row in reader:
        print(f"{row['model']:12s}: acc={float(row['accuracy']):.2%}")

# In ML we almost always use Pandas for CSV - much more powerful
import pandas as pd
df = pd.read_csv('model_results.csv')
print('\nLoaded with Pandas:')
print(df.to_string(index=False))
print(f'Best model: {df.loc[df.accuracy.idxmax(), "model"]}')

**Example 3.7.3**

In [ ]:
import json
# JSON for hyperparameter configs and experiment results
experiment = {
    'experiment_id': 'exp_2024_001',
    'model': 'ResNet50',
    'hyperparameters': {
        'learning_rate': 0.001,
        'batch_size': 32,
        'n_epochs': 100,
        'optimizer': 'adam',
        'weight_decay': 1e-4
    },
    'results': {
        'train_accuracy': 0.9823,
        'val_accuracy': 0.9654,
        'test_accuracy': 0.9612,
        'training_time_hours': 4.5
    },
    'feature_names': ['pixel_mean', 'pixel_std', 'edge_density'],
    'converged': True
}

# Write JSON
with open('experiment.json', 'w') as f:
    json.dump(experiment, f, indent=2)   # indent=2 for readable formatting

# Read JSON
with open('experiment.json', 'r') as f:
    loaded = json.load(f)

print(f'Experiment: {loaded["experiment_id"]}')
print(f'Test accuracy: {loaded["results"]["test_accuracy"]:.1%}')
print(f'Best LR: {loaded["hyperparameters"]["learning_rate"]}')

# JSON string conversion (useful for APIs and logging)
config_str = json.dumps(experiment['hyperparameters'])
print(f'Config JSON: {config_str}')

**Example 3.7.4**

In [ ]:
import pickle
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

# Train a model
X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
print(f'Trained accuracy: {model.score(X_test, y_test):.4f}')

# Save model to disk
with open('iris_rf_model.pkl', 'wb') as f:   # 'wb' = write binary
    pickle.dump(model, f)
print('Model saved to iris_rf_model.pkl')

# Load model from disk
with open('iris_rf_model.pkl', 'rb') as f:   # 'rb' = read binary
    loaded_model = pickle.load(f)

# Verify loaded model works identically
print(f'Loaded accuracy: {loaded_model.score(X_test, y_test):.4f}')

# joblib is faster than pickle for large NumPy arrays (preferred for sklearn)
import joblib
joblib.dump(model, 'iris_rf_model.joblib')
loaded_jl = joblib.load('iris_rf_model.joblib')
print(f'joblib accuracy: {loaded_jl.score(X_test, y_test):.4f}')